# Load minIO Landing to MinIO bronze (Incremental)

### Install python dotenv for get the environment variables

In [8]:
pip install python-dotenv

IOStream.flush timed out
Note: you may need to restart the kernel to use updated packages.


## Imports libs, files and configure the absolute path

In [9]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession, functions
from pyspark.sql.functions import date_format, col, row_number
from pyspark.sql.window import Window
import logging
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Import for get the environment variables 
from dotenv import load_dotenv
import os
import sys
sys.path.append(os.path.abspath("../../")) # Used to reconfigure the absolut path. In this case, setting the absolut path to 2 folders back (notebooks/...) 
from configurations import configurations as config_file # Import configurations.py from the configurations folder
from functions import functions as func_file # Import functions.py from the functions folder

## Load environment variables

In [10]:
load_dotenv()

MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")
POSTGRES_CONTAINER=os.getenv("POSTGRES_CONTAINER")
POSTGRES_USER=os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD=os.getenv("POSTGRES_PASSWORD")

## Spark configurations

In [11]:
def configure_spark():
    conf = SparkConf()
    
    conf.setAppName("Incremental load from MinIO landing to MinIO bronze") # Spark application name, Usefull for logs
    conf.set("spark.master", "spark://spark-master:7077") # set the Spark container to be distributed among the workers
    conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
    conf.set("spark.hadoop.fs.s3a.access.key", MINIO_USER) # Login from MinIO
    conf.set("spark.hadoop.fs.s3a.secret.key", MINIO_PASSWORD) # Password from MinIO
    # Add the jars from hadoop-aws and aws-java-sdk-bundle is necessary for org.apache.hadoop.fs.s3a.S3AFileSystem,
    # add the Postgresql JDBC jar is necessary for connect on database. Add the delta-spark is necessary for delta catalog, all this Jars is auto-download from spark
    conf.set("spark.jars.packages", 
             "org.apache.hadoop:hadoop-aws:3.3.4,"
             "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
             "org.postgresql:postgresql:42.7.2,"
             "io.delta:delta-spark_2.12:3.1.0" )
    conf.set("spark.hadoop.fs.s3a.path.style.access", True) # Enforces the use of URLs as the format. Without this, Spark attempts to use the AWS standard (bucket.endpoint), which fails in MinIO
    conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
    conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
    conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
    conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external
    
    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    return spark

## Configuration the Logging and log the startup

In [14]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

## Creating a method to process each table when is called on main method

In [15]:
def process_table(spark, table_name, table_name_converted, primary_key, input_table_path, output_table_path):

    try:
    
        # Logging the processing
        logging.info(f"processing table {table_name_converted}")

        # 1. Tratamento para Primeira Carga (Se a tabela Bronze não existir)
        try:
            # Getting max date value from minIO bronze in the modifieddate column. limit at 1 result and get this result on 1º row at max_modifieddate column
            df_max_modifieddate_bronze = spark.read.format("delta").load(output_table_path) \
                .select(functions.max("modifieddate").alias("max_modifieddate")).limit(1).collect()[0]["max_modifieddate"]
            
            # Se a tabela existir mas estiver vazia (retornar None)
            if df_max_modifieddate_bronze is None:
                df_max_modifieddate_bronze = "1900-01-01 00:00:00"
                
        except Exception:
            
            # Caso o caminho não exista no MinIO (Primeira execução)
            logging.info(f"Bronze table for {table_name_converted} not found. Starting initial load.")
            df_max_modifieddate_bronze = "1900-01-01 00:00:00"

        
        # Getting max date value from minIO landing on parquet format in the modifieddate column.
        df_update_data_to_bronze = spark.read.format("parquet").load(input_table_path) \
        .filter(functions.col("modifieddate") > functions.lit(df_max_modifieddate_bronze))
        
        rows_to_update = df_update_data_to_bronze.count()
        
        if  rows_to_update == 0:
            # Logging if get no rows to update in minio landing
            logging.info(f"No new data to process for table {table_name_converted}")

        else:
            # Logging number of rows to update
            logging.info(f"Number of new rows to update for table {table_name_converted}: {rows_to_update}")

            # Deduplicação do lote incremental usando Window Function
            # Evita o erro de linhas ambíguas no Merge caso o mesmo ID tenha mudado mais de uma vez na Landing
            #window_spec = Window.partitionBy(primary_key).orderBy(col("modifieddate").desc())
            #df_deduplicated_batch = df_update_data_to_bronze \
            #    .withColumn("row_num", row_number().over(window_spec)) \
            #    .filter(col("row_num") == 1) \
            #    .drop("row_num")

            # 1. Definimos o conjunto de colunas técnicas e metadados para ignorar
            ignored_columns = {primary_key, "modifieddate", "month_key", "last_update"}

            # Pegamos todas as colunas menos metadados técnicos para gerar o hash
            business_columns = [functions.col(c) for c in df_update_data_to_bronze.columns if c not in ignored_columns]

            # 3. Criamos a expressão do hash separadamente para não poluir o .withColumn
            # o "*" em Python é chamado de desempactador (Unpacking Operator), basicamente ele faz com que abra a lista passando cada elemento
            # da lista separados por vírgulas como é o caso do "CONCAT_WS" que espera receber todos os elementos seguintes separados por vírgula
            row_hash_expr = functions.sha2(functions.concat_ws("||", *business_columns), 256)
            
            # Gerando o hash com base nas colunas de negócio reais
            df_with_hash = df_update_data_to_bronze.withColumn("row_hash", row_hash_expr)
                
            # Adding a new column date related the load data
            df_with_update_date = func_file.add_data_last_update(df_with_hash)
    
            # modifing the dataframe to add a new column "month_key" to create a partition on the minIO Bronze based on modifieddate column
            df_with_month_partition = df_with_update_date.withColumn("month_key", date_format(df_with_update_date["modifieddate"], "yyyy-MM"))
            
            # Updating the dataframe on minIO landing
            logging.info(f"Updating table {table_name_converted}...")


            try:
                # Instanciar a tabela bronze atual mapeada no MinIO como um objeto DeltaTable
                target_delta_table = DeltaTable.forPath(spark, output_table_path)
                
                # Executar o Merge Inteligente
                target_delta_table.alias("bronze") \
                    .merge(
                        source=df_with_month_partition.alias("updates"),
                        # Condição 1: Casar pelo ID do registro
                        condition=f"bronze.{primary_key} = updates.{primary_key}"
                    ) \
                    .whenMatchedUpdate(
                        # Condição 2: Só atualiza a Bronze se o hash dos dados novos for DIFERENTE do hash antigo
                        condition="bronze.row_hash != updates.row_hash",
                        set={
                            # Atualiza todas as colunas com os valores novos do lote
                            **{c: f"updates.{c}" for c in df_with_month_partition.columns}
                        }
                    ) \
                    .whenNotMatchedInsert(
                        # Condição 3: Se o ID não existe na Bronze, insere como um novo registro
                        values={c: f"updates.{c}" for c in df_with_month_partition.columns}
                    ) \
                    .execute()
                
            except Exception:
                
                # Fallback para criar a tabela na primeira carga caso o DeltaTable.forPath falhe por falta de diretório
                logging.info(f"Creating Delta table for the first time on path: {output_table_path}")
                df_with_month_partition.write.format("delta").mode("overwrite").partitionBy("month_key").save(output_table_path)


            # Logging the sucessfully process
            logging.info(f"Table {table_name_converted} Sucessfully updated and saved in MinIO bronze on: {output_table_path}")

    except Exception as e:
        # Logging the Error
         logging.error(f"Error processing table {table_name}: {str(e)}")




2026-05-29 00:32:22,281 - INFO - Error while sending or receiving.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer
2026-05-29 00:32:22,285 - INFO - Closing down clientserver connection
2026-05-29 00:32:22,287 - INFO - Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^

In [16]:
if __name__ == "__main__":
    
    # Logging the Start process from ingestion
    logging.info("Starting incrmental load from MinIO landing to MinIO bronze...")

    spark = configure_spark()

    # landing path
    landing_path = config_file.data_lakehouse_path["landing"]
    
    # bronze path
    bronze_path = config_file.data_lakehouse_path["bronze"]

    # Dictionary with primary keys from tables
    dictionary_pks = config_file.tables_pk

    # Creating a ThreadPool for divide all jobs among the workers and execute in parallel
    with ThreadPoolExecutor(max_workers=8) as executor:
        
        # Creating a list to Add all jobs into it
        futures = []
    
        # Getting each table was in the dictionary on config_file
        for table_name in config_file.tables_postgres_adventureworks.values():
            
            # convert the table name from postgres to name in minIO s3
            table_name_converted = func_file.convert_table_name(table_name)
    
            # Landing table path
            landing_table_path = f"{landing_path}{table_name_converted}"
            
            # Output table path
            bronze_table_path = f"{bronze_path}bronze_{table_name_converted}"

            #Primary key from table_name
            primary_key = func_file.get_pk(table_name_converted, dictionary_pks)

            # Instead of calling the function to execute it, call the function by passing it to the executor
            futures.append(executor.submit(process_table, spark, table_name, table_name_converted, primary_key, landing_table_path, bronze_table_path))


        for future in as_completed(futures):
            try:
                # Where the all jobs is executing by the executors created with ThreadPoolExecutor
                future.result()

            except Exception as e:
                logging.error(f"Error in one of parallel taks: {str(e)}")

    
    # Logging the Incremental ingestion
    logging.info(f"Incremental ingestion to bronze layer completed!")
    

2026-05-29 00:32:32,362 - INFO - Starting incrmental load from MinIO landing to MinIO bronze...
2026-05-29 00:32:33,227 - INFO - processing table sales_countryregioncurrency
2026-05-29 00:32:33,228 - INFO - processing table sales_creditcard
2026-05-29 00:32:33,229 - INFO - processing table sales_currency
2026-05-29 00:32:33,231 - INFO - processing table sales_currencyrate
2026-05-29 00:32:33,231 - INFO - processing table sales_customer
2026-05-29 00:32:33,232 - INFO - processing table sales_personcreditcard
2026-05-29 00:32:33,233 - INFO - processing table sales_salesorderdetail
2026-05-29 00:32:33,235 - INFO - processing table sales_salesorderheader
2026-05-29 00:32:33,477 - INFO - Bronze table for sales_creditcard not found. Starting initial load.
2026-05-29 00:32:33,481 - INFO - Bronze table for sales_salesorderdetail not found. Starting initial load.
2026-05-29 00:32:33,495 - INFO - Bronze table for sales_countryregioncurrency not found. Starting initial load.
2026-05-29 00:32:33,5

## Stop session and clear cash from spark

In [17]:
spark.stop()
spark.catalog.clearCache()